# Part 7b U55C: Deployment

Run this notebook on a HACC U55C node after Part 7a has produced the staged Coyote build.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np

WORKSPACE = Path.cwd()
ML_BASELINE = WORKSPACE.parent
COYOTE_ROOT = Path("/pub/scratch/sdeheredia/Coyote")
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))
if str(ML_BASELINE) not in sys.path:
    sys.path.insert(0, str(ML_BASELINE))

DEFAULT_ITERATION_ROOT = WORKSPACE / "artifacts/cnn_small_hls_opt_img512/notebook_pruned_qat/pruned_qat_w6_a6_s50_rf8_0cfa065db9d4"
DEFAULT_FOLD = 0
DEFAULT_HLS_SWEEP = "hls_1e918f3210d4"
ABI = {
    "img_size": 512,
    "pixels_per_sample": 512 * 512,
    "axi_data_bits": 512,
    "fixed_width": 16,
    "fixed_integer": 6,
    "fixed_fraction": 10,
    "pixels_per_beat": 32,
    "beats_per_sample": 8192,
    "input_bytes_per_sample": 512 * 512 * 2,
    "output_bytes_per_sample": 64,
}

def read_json(path: Path) -> dict:
    return json.loads(Path(path).read_text())

def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True))

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_tree(root: Path, patterns=(".cpp", ".h", ".hpp", ".svh", ".txt", ".cmake", "CMakeLists.txt")) -> str:
    h = hashlib.sha256()
    root = Path(root)
    if not root.exists():
        return ""
    for path in sorted(p for p in root.rglob("*") if p.is_file()):
        if path.name == "CMakeLists.txt" or any(path.name.endswith(pat) for pat in patterns):
            h.update(str(path.relative_to(root)).encode())
            h.update(path.read_bytes())
    return h.hexdigest()

def run(cmd: list[str], cwd: Path, log_path: Path | None = None) -> subprocess.CompletedProcess:
    print("$", " ".join(map(str, cmd)))
    proc = subprocess.run(cmd, cwd=str(cwd), text=True, capture_output=True)
    if log_path is not None:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text("STDOUT\n" + proc.stdout + "\nSTDERR\n" + proc.stderr)
    if proc.stdout:
        print(proc.stdout[-4000:])
    if proc.stderr:
        print(proc.stderr[-4000:])
    proc.check_returncode()
    return proc

def sigmoid(x):
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-x))

def clean_rows(csv_path: Path) -> list[dict]:
    if not Path(csv_path).exists():
        return []
    with Path(csv_path).open(newline="") as handle:
        rows = [row for row in csv.DictReader(handle)]
    return [row for row in rows if row.get("sample_index") not in ("sample_index", "", None)]

def binary_loss(label: float, prob: float) -> float:
    p = float(np.clip(prob, 1e-7, 1.0 - 1e-7))
    return float(-(label * np.log(p) + (1.0 - label) * np.log(1.0 - p)))

def rows_from_logits(meta_rows: list[dict], logits: np.ndarray) -> list[dict]:
    probs = sigmoid(logits)
    rows = []
    for i, (meta, logit, prob) in enumerate(zip(meta_rows, logits, probs)):
        label = int(meta["class_label"])
        pred = int(prob >= 0.5)
        rows.append({
            "sample_index": i,
            "sample_id": meta.get("sample_id", ""),
            "app_name": meta.get("app_name", ""),
            "class_label": label,
            "class_name": meta.get("class_name", "standalone" if label else "benign"),
            "ro_count": meta.get("ro_count", ""),
            "bitstream_path": meta.get("bitstream_path", ""),
            "logit": f"{float(logit):.9f}",
            "probability": f"{float(prob):.9f}",
            "predicted_label": pred,
            "correct": int(pred == label),
            "per_sample_bce_loss": f"{binary_loss(label, prob):.9f}",
            "per_sample_log_loss": f"{binary_loss(label, prob):.9f}",
        })
    return rows

def write_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text("")
        return
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


In [ ]:
ITERATION_ROOT = DEFAULT_ITERATION_ROOT
SELECTED_FOLD = DEFAULT_FOLD
REQUESTED_HLS_SWEEP = DEFAULT_HLS_SWEEP

def discover_sweeps(iteration_root: Path, fold: int) -> list[dict]:
    out = []
    for sweep in sorted((iteration_root / "hls_sweeps").glob("*")):
        project = sweep / f"fold_{fold}" / "project"
        conv = project / "conversion_manifest.json"
        if conv.exists() and (project / "firmware").is_dir():
            payload = read_json(conv)
            out.append({
                "name": sweep.name,
                "path": sweep,
                "project": project,
                "project_name": payload["project_name"],
                "hls_fingerprint": payload.get("hls_fingerprint", ""),
            })
    return out

sweeps = discover_sweeps(ITERATION_ROOT, SELECTED_FOLD)
if not sweeps:
    raise FileNotFoundError(f"No generated hls4ml projects found below {ITERATION_ROOT}")
selected = next((s for s in sweeps if s["name"] == REQUESTED_HLS_SWEEP), sweeps[0])

HLS_SWEEP_ROOT = selected["path"]
HLS_PROJECT_DIR = selected["project"]
PROJECT_NAME = selected["project_name"]
FOLD_DIR = ITERATION_ROOT / f"fold_{SELECTED_FOLD}"
PARITY_DIR = HLS_SWEEP_ROOT / f"fold_{SELECTED_FOLD}" / "parity"
U55C_ROOT = HLS_SWEEP_ROOT / f"fold_{SELECTED_FOLD}" / "u55c_deployment"
PREPARED_INPUTS_DIR = U55C_ROOT / "prepared_inputs"
STAGED_HW_DIR = U55C_ROOT / "coyote_hw"
STAGED_SW_DIR = U55C_ROOT / "coyote_sw"
VALIDATION_DIR = HLS_SWEEP_ROOT / f"fold_{SELECTED_FOLD}" / "u55c_validation"

print("iteration root:", ITERATION_ROOT)
print("selected fold:", SELECTED_FOLD)
print("selected sweep:", selected["name"])
print("hls project:", HLS_PROJECT_DIR)
print("artifact root:", U55C_ROOT)
print("available sweeps:", [s["name"] for s in sweeps])


In [ ]:
bit_manifest_path = U55C_ROOT / "bitstream_manifest.json"
if not bit_manifest_path.exists():
    raise FileNotFoundError(f"Run Part 7a first: {bit_manifest_path}")
bit_manifest = read_json(bit_manifest_path)
bitstreams = [Path(p) for p in bit_manifest.get("bitstream_candidates", []) if Path(p).exists()]
if not bitstreams:
    print("No built bitstream recorded yet. Part 7a must complete make bitgen before programming.")
else:
    print("selected bitstream:", bitstreams[-1])
print("prepared manifest:", PREPARED_INPUTS_DIR / "manifest.csv")
print("software dir:", STAGED_SW_DIR)


## Build Host Harness


In [ ]:
sw_build = STAGED_SW_DIR / "build"
sw_build.mkdir(parents=True, exist_ok=True)
sw_fingerprint = {
    "sw_source_hash": sha256_tree(STAGED_SW_DIR),
    "coyote_root": str(COYOTE_ROOT),
    "prepared_manifest_hash": sha256_file(PREPARED_INPUTS_DIR / "manifest.csv"),
}
deployment_manifest_path = U55C_ROOT / "deployment_manifest.json"
deployment_manifest = read_json(deployment_manifest_path) if deployment_manifest_path.exists() else {}
host_exe = sw_build / "coyote_qkeras_host"
if deployment_manifest.get("sw_fingerprint") != sw_fingerprint or not host_exe.exists():
    run(["cmake", ".."], cwd=sw_build, log_path=U55C_ROOT / "logs" / "cmake_sw.log")
    run(["make", "-j", str(os.cpu_count() or 4)], cwd=sw_build, log_path=U55C_ROOT / "logs" / "make_sw.log")
    deployment_manifest["sw_fingerprint"] = sw_fingerprint
    deployment_manifest["host_executable"] = str(host_exe)
    write_json(deployment_manifest_path, deployment_manifest)
else:
    print("host harness cache hit")
print("host executable:", host_exe)


## Program U55C And Run Validation Samples


In [ ]:
bitstreams = [Path(p) for p in bit_manifest.get("bitstream_candidates", []) if Path(p).exists()]
if not bitstreams:
    raise FileNotFoundError("No bitstream available from Part 7a")
bitstream = bitstreams[-1]
driver_candidates = sorted(COYOTE_ROOT.rglob("coyote_driver.ko"))
if not driver_candidates:
    raise FileNotFoundError("Could not find coyote_driver.ko below Coyote root")
driver = driver_candidates[-1]
program_script = COYOTE_ROOT / "util" / "program_hacc_local.sh"
if not program_script.exists():
    raise FileNotFoundError(program_script)

run([str(program_script), str(bitstream), str(driver)], cwd=COYOTE_ROOT, log_path=U55C_ROOT / "logs" / "program_u55c.log")

hardware_csv = U55C_ROOT / "hardware_per_sample.csv"
proc = run([
    str(host_exe),
    "--manifest", str(PREPARED_INPUTS_DIR / "manifest.csv"),
    "--output", str(hardware_csv),
], cwd=sw_build, log_path=U55C_ROOT / "logs" / "host_run.log")

rows = clean_rows(hardware_csv)
logits = np.asarray([float(row["logit"]) for row in rows], dtype=np.float32)
lat = np.asarray([float(row["latency_us"]) for row in rows], dtype=np.float64)
np.save(U55C_ROOT / "y_hw.npy", logits)
latency_summary = {
    "n_samples": int(len(rows)),
    "latency_us_mean": float(np.mean(lat)) if len(lat) else None,
    "latency_us_median": float(np.median(lat)) if len(lat) else None,
    "latency_us_min": float(np.min(lat)) if len(lat) else None,
    "latency_us_max": float(np.max(lat)) if len(lat) else None,
    "throughput_samples_per_s": float(1e6 / np.mean(lat)) if len(lat) else None,
}
write_json(U55C_ROOT / "latency_summary.json", latency_summary)

deployment_manifest.update({
    "deployed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "bitstream": str(bitstream),
    "driver": str(driver),
    "hardware_per_sample_csv": str(hardware_csv),
    "y_hw": str(U55C_ROOT / "y_hw.npy"),
    "latency_summary": latency_summary,
})
write_json(deployment_manifest_path, deployment_manifest)

print(json.dumps(latency_summary, indent=2))
print("hardware csv:", hardware_csv)
print("y_hw:", U55C_ROOT / "y_hw.npy")
